In [ ]:
import pandas as pd
import numpy as np
import os

STD_PATH = "./Standardized_Data"
FREQS = ['1min','5min', '15min'] 

def aggregation_with_diffs(file_name):
    full_path = os.path.join(STD_PATH, file_name)
    
    for f in FREQS:
        output_dir = os.path.join("./Data", f"Aggregated_{f}")
        os.makedirs(output_dir, exist_ok=True)
        output_file = os.path.join(output_dir, f"agg_{f}_{file_name}")

        print(f"Agg: {f}: {file_name} (price_diff/return)...")
        results = []
        reader = pd.read_csv(full_path, chunksize=1000000)
        
        for chunk in reader:
            chunk['timestamp'] = pd.to_datetime(chunk['timestamp'], format='ISO8601', utc=True)
            chunk['dollar_vol'] = chunk['price'] * chunk['amount']
            chunk['signed_vol'] = np.where(chunk['side'] == 'buy', chunk['amount'], -chunk['amount'])
            
            agg = chunk.groupby(pd.Grouper(key='timestamp', freq=f)).agg(
                open=('price', 'first'),
                high=('price', 'max'),
                low=('price', 'min'),
                close=('price', 'last'),
                volume=('amount', 'sum'),
                dollar_volume=('dollar_vol', 'sum'),
                signed_vol=('signed_vol', 'sum')
            )
            results.append(agg)
            
        final_df = pd.concat(results).groupby(level=0).agg({
            'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
            'volume': 'sum', 'dollar_volume': 'sum', 'signed_vol': 'sum'
        })
        
        final_df['close'] = final_df['close'].ffill()
        final_df['price_diff'] = final_df['close'].diff().fillna(0)
        final_df['return'] = final_df['close'].pct_change().fillna(0)
        
        final_df[['open', 'high', 'low']] = final_df[['open', 'high', 'low']].fillna(method='ffill', axis=1)
        final_df[['volume', 'dollar_volume', 'signed_vol']] = final_df[['volume', 'dollar_volume', 'signed_vol']].fillna(0)
        
        final_df.to_csv(output_file)

for f_name in os.listdir(STD_PATH):
    if f_name.startswith('std_'):
        aggregation_with_diffs(f_name)


import pandas as pd
import numpy as np
import os
from scipy import stats


BASE_DATA_PATH = "./Data"
FREQS = ['1min', '5min', '15min']

def compute_metrics_for_all_freqs():
    for freq in FREQS:
        input_dir = os.path.join(BASE_DATA_PATH, f"Aggregated_{freq}")
        output_dir = os.path.join(BASE_DATA_PATH, f"Processed_Metrics_{freq}")
        os.makedirs(output_dir, exist_ok=True)
        
        if not os.path.exists(input_dir):
            continue
            
        
        for f_name in os.listdir(input_dir):
            if not f_name.startswith('agg_'): continue
            
            df = pd.read_csv(os.path.join(input_dir, f_name), index_col=0, parse_dates=True)
            
       
            price_diff = df['close'].diff()
            roll_cov = price_diff.rolling(window=30).cov(price_diff.shift(1))
            df['roll_spread'] = 2 * np.sqrt(np.maximum(-roll_cov, 0))
            df['cs_proxy'] = np.log(df['high'] / df['low'])

            
            df['amihud'] = df['return'].abs() / (df['dollar_volume'] / 1e6 + 1e-9)
            
            window_size = 30 if freq == '1min' else 12
            std_p = price_diff.rolling(window_size).std()
            std_v = df['signed_vol'].rolling(window_size).std()
            corr = price_diff.rolling(window_size).corr(df['signed_vol'])
            df['kyle_lambda'] = corr * (std_p / (std_v + 1e-9))

            
            df['rv'] = df['return'].rolling(window=10).apply(lambda x: np.sqrt(np.sum(x**2)))

            
            if freq == '1min':
                df['is_gap'] = (df['volume'] == 0).astype(int)

            df.to_csv(os.path.join(output_dir, f"metrics_{f_name}"))

compute_metrics_for_all_freqs()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

METRICS_PATH = "./Data/Processed_Metrics_15min"

target_files = {
    'Kraken BTC/USD': 'metrics_agg_15min_std_kraken_btc_usd_march_2023.csv',
    'Binance BTC/USDT': 'metrics_agg_15min_std_BTCUSDT-trades-2023-03us.csv',
    'Kraken BTC/USDC': 'metrics_agg_15min_std_kraken_btc_usdc_march_2023.csv'
}

plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 18), sharex=True)

colors = {'Kraken BTC/USD': '#1f77b4', 'Binance BTC/USDT': '#ff7f0e', 'Kraken BTC/USDC': '#2ca02c'}

for label, f_name in target_files.items():
    file_path = os.path.join(METRICS_PATH, f_name)
    if not os.path.exists(file_path):
        print(f"can't find file: {f_name}")
        continue
        
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    df_plot = df.loc['2023-03-01':'2023-03-21']

    ax1.plot(df_plot.index, df_plot['roll_spread'], label=label, alpha=0.8, color=colors[label])
    
    ax2.plot(df_plot.index, df_plot['kyle_lambda'], label=label, alpha=0.8, color=colors[label])
    
    ax3.plot(df_plot.index, df_plot['rv'], label=label, alpha=0.8, color=colors[label])

for ax in [ax1, ax2, ax3]:
    ax.axvspan('2023-03-09', '2023-03-12', color='gray', alpha=0.15, label='Crisis Window')
    ax.legend(loc='upper left', frameon=True)
    ax.grid(True, linestyle='--', alpha=0.6)

ax1.set_title('Dimension 1: Transaction Cost (Roll Spread)', fontsize=16, fontweight='bold')
ax1.set_ylabel('Spread Value', fontsize=12)

ax2.set_title("Dimension 2: Price Impact (Kyle's Lambda)", fontsize=16, fontweight='bold')
ax2.set_ylabel('Lambda (Price Sensitivity)', fontsize=12)

ax3.set_title('Dimension 3: Price Volatility (Realized Volatility)', fontsize=16, fontweight='bold')
ax3.set_ylabel('RV (%)', fontsize=12)

plt.xlabel('Date (March 2023)', fontsize=12)
plt.tight_layout()
plt.savefig("Full_Sample_Metrics_Lambda.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

files = {
    'Kraken BTC/USD (Onshore Fiat)': 'cm_hourly_kraken-btc-usd-spot.csv',
    'Binance BTC/USDT (Offshore Stable)': 'cm_hourly_binance-btc-usdt-spot.csv',
    'Kraken BTC/USDC (Depeg Epicenter)': 'cm_hourly_kraken-btc-usdc-spot.csv'
}

dataframes = {}
for label, filename in files.items():
    if os.path.exists(filename):
        df = pd.read_csv(filename, index_col=0, parse_dates=True)
        dataframes[label] = df.loc['2023-03-09':'2023-03-14'].copy()

if len(dataframes) > 0:
    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(18, 12), sharex=True)
    
    colors = ['#1f77b4', '#ff7f0e', '#d62728']
    
    for i, (label, df) in enumerate(dataframes.items()):
        c = colors[i]
        
        ax_slip = axes[i, 0]
        if 'liquidity_slippage_1M_ask_percent' in df.columns:
            ax_slip.plot(df.index, df['liquidity_slippage_1M_ask_percent'], color=c, linewidth=2)
        ax_slip.set_title(f"{label}\n1M Ask Slippage (%)", fontsize=12, fontweight='bold')
        ax_slip.set_ylabel("Slippage (%)", fontsize=11)
        ax_slip.grid(True, linestyle='--', alpha=0.5)
        ax_slip.axvspan('2023-03-11 00:00', '2023-03-12 12:00', color='red', alpha=0.1)

        ax_depth = axes[i, 1]
        if 'liquidity_depth_1_percent_bid_volume_usd' in df.columns:
            ax_depth.plot(df.index, df['liquidity_depth_1_percent_bid_volume_usd'] / 1e6, color=c, linewidth=2)
        ax_depth.set_title(f"{label}\n1% Bid Depth (Million USD)", fontsize=12, fontweight='bold')
        ax_depth.set_ylabel("Depth (M USD)", fontsize=11)
        ax_depth.grid(True, linestyle='--', alpha=0.5)
        ax_depth.axvspan('2023-03-11 00:00', '2023-03-12 12:00', color='red', alpha=0.1)

    axes[2, 0].set_xlabel('Date (UTC)', fontsize=12)
    axes[2, 1].set_xlabel('Date (UTC)', fontsize=12)
    fig.suptitle('Order Book Microstructure Shock (March 9 - March 14, 2023)', 
                 fontsize=20, fontweight='bold', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig("CoinMetrics_Matrix_Shock.png", dpi=300, bbox_inches='tight')
    print("generated CoinMetrics_Matrix_Shock.png")
    plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

METRICS_PATH = "./Data/Processed_Metrics_5min"

files = {
    'Kraken BTC/USD': 'metrics_agg_5min_std_kraken_btc_usd_march_2023.csv',
    'Binance BTC/USDT': 'metrics_agg_5min_std_BTCUSDT-trades-2023-03com.csv'
}

df_kraken = pd.read_csv(os.path.join(METRICS_PATH, files['Kraken BTC/USD']), index_col=0, parse_dates=True)
df_binance = pd.read_csv(os.path.join(METRICS_PATH, files['Binance BTC/USDT']), index_col=0, parse_dates=True)

df_k_plot = df_kraken.loc['2023-03-09':'2023-03-12'].copy()
df_b_plot = df_binance.loc['2023-03-09':'2023-03-12'].copy()

window_size = 6
df_k_plot['roll_smooth'] = df_k_plot['roll_spread'].rolling(window_size, min_periods=1).mean()
df_b_plot['roll_smooth'] = df_b_plot['roll_spread'].rolling(window_size, min_periods=1).mean()
df_k_plot['cs_smooth'] = df_k_plot['cs_proxy'].rolling(window_size, min_periods=1).mean()
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax1.plot(df_k_plot.index, df_k_plot['roll_smooth'], label='Kraken BTC/USD (Onshore)', color='#1f77b4', linewidth=2)
ax1.plot(df_b_plot.index, df_b_plot['roll_smooth'], label='Binance BTC/USDT (Offshore)', color='#ff7f0e', linewidth=2, alpha=0.8)
ax1.set_title('1.1 Roll Spread: The "Zero Trap" and One-Sided Panic (March 09-12)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Roll Spread', fontsize=12)
ax1.legend(loc='upper right')
ax1.grid(True, linestyle='--', alpha=0.5)

ax1.axvspan('2023-03-09 20:00', '2023-03-11 12:00', color='red', alpha=0.1, label='Extreme Panic (Roll Fails)')

ax2.plot(df_k_plot.index, df_k_plot['cs_smooth'], label='Kraken BTC/USD CS Proxy', color='#d62728', linewidth=2)
ax2.set_title('1.2 Corwin-Schultz Spread: Capturing True Friction When Roll Fails', fontsize=14, fontweight='bold')
ax2.set_ylabel('CS Spread (High-Low)', fontsize=12)
ax2.set_xlabel('Date (UTC)', fontsize=12)
ax2.legend(loc='upper right')
ax2.grid(True, linestyle='--', alpha=0.5)

ax2.axvspan('2023-03-09 20:00', '2023-03-11 12:00', color='red', alpha=0.1)

plt.tight_layout()
plt.savefig("DeepDive_Width_Roll_vs_CS.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

METRICS_PATH = "./Data/Processed_Metrics_5min"

files = {
    'Kraken BTC/USDC': 'metrics_agg_5min_std_kraken_btc_usdc_march_2023.csv',
    'Kraken BTC/USDT': 'metrics_agg_5min_std_kraken_btc_usdt_march_2023.csv'
}

data = {}
for label, f_name in files.items():
    file_path = os.path.join(METRICS_PATH, f_name)
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    if df.index.tz is None:
        df.index = df.index.tz_localize('UTC')
    data[label] = df.loc['2023-03-09':'2023-03-12'].copy()

df_usdc = data['Kraken BTC/USDC']
df_usdt = data['Kraken BTC/USDT']

window_size = 6
df_usdc['amihud_smooth'] = df_usdc['amihud'].rolling(window=window_size, min_periods=1).mean()
df_usdt['amihud_smooth'] = df_usdt['amihud'].rolling(window=window_size, min_periods=1).mean()
df_usdc['lambda_smooth'] = df_usdc['kyle_lambda'].rolling(window=window_size, min_periods=1).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax1.plot(df_usdc.index, df_usdc['amihud_smooth'] * 1e8, label='Kraken BTC/USDC (Depeg Epicenter)', color='#d62728', linewidth=2)
ax1.plot(df_usdt.index, df_usdt['amihud_smooth'] * 1e8, label='Kraken BTC/USDT (Control Group)', color='#1f77b4', linewidth=2, alpha=0.8)

ax1.set_title('2.1 Amihud Illiquidity: USDC vs USDT on Kraken (Scaled by 1e8)', fontsize=15, fontweight='bold')
ax1.set_ylabel('Amihud ILLIQ', fontsize=12)
ax1.legend(loc='upper left')
ax1.grid(True, linestyle='--', alpha=0.5)

crisis_start = pd.to_datetime('2023-03-11 00:00:00').tz_localize('UTC')
crisis_end = pd.to_datetime('2023-03-12 12:00:00').tz_localize('UTC')
ax1.axvspan(crisis_start, crisis_end, color='gray', alpha=0.15)
color_lambda = '#ff7f0e'
color_vol = '#2ca02c'

ax2.set_title('2.2 Price Impact (Lambda) vs. Trading Volume (Kraken BTC/USDC)', fontsize=15, fontweight='bold')
ax2.set_xlabel('Date (UTC)', fontsize=12)

ax2.set_ylabel('Kyle\'s Lambda (Smoothed)', color=color_lambda, fontsize=12)
ax2.plot(df_usdc.index, df_usdc['lambda_smooth'], color=color_lambda, linewidth=2.5, label='Lambda (Price Impact)')
ax2.tick_params(axis='y', labelcolor=color_lambda)
ax3 = ax2.twinx()
ax3.set_ylabel('Dollar Volume (USD)', color=color_vol, fontsize=12)
ax3.bar(df_usdc.index, df_usdc['dollar_volume'], width=0.005, color=color_vol, alpha=0.3, label='Trading Volume')
ax3.tick_params(axis='y', labelcolor=color_vol)

ax2.axvspan(crisis_start, crisis_end, color='gray', alpha=0.15)

lines_1, labels_1 = ax2.get_legend_handles_labels()
lines_2, labels_2 = ax3.get_legend_handles_labels()
ax2.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.tight_layout()
plt.savefig("DeepDive_Depth_5min.png", dpi=300)
print("generated DeepDive_Depth_5min.png")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

METRICS_PATH = r"C:\Users\Yvaine\Desktop\IAQF\Data\Processed_Metrics_1min"
file_name = 'metrics_agg_1min_std_kraken_btc_usd_march_2023.csv'
file_path = os.path.join(METRICS_PATH, file_name)

try:
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
except Exception as e:
    print(f"fail to read file: {e}")
    exit()

if df.index.tz is None:
    df.index = df.index.tz_localize('UTC')
else:
    df.index = df.index.tz_convert('UTC')

print(f"earliest time: {df.index.min()}")
print(f"latest time: {df.index.max()}")

try:
    df_plot = df.loc['2023-03-10':'2023-03-13'].copy()
except Exception as e:
    print(f"fail to cut time, please check your csv file: {e}")
    exit()
volume_col = 'dollar_volume'
if volume_col not in df_plot.columns:
    possible_cols = [c for c in df_plot.columns if 'volume' in c.lower()]
    if possible_cols:
        volume_col = possible_cols[0]
        print(f"no 'dollar_volume' found, auto replace to: '{volume_col}'")
    else:
        print(f"no volume column found, current columns: {df_plot.columns.tolist()}")
        exit()
hourly_volume = df_plot[volume_col].resample('1h').sum() / 1e6

print(f"aggregated volume: {hourly_volume.max():.2f} ")

if hourly_volume.sum() == 0:
    print("warning: your volume data is all 0!")
else:
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(hourly_volume.index, hourly_volume.values, width=1/24, color='#1f77b4', alpha=0.8, edgecolor='black', linewidth=0.5)

    ax.set_title('Dimension 4: The TradFi "Settlement Basin" (Hourly Volume)', fontsize=16, fontweight='bold')
    ax.set_ylabel('Hourly Volume (Million USD)', fontsize=12)
    ax.set_xlabel('Date (UTC)', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.5)

    bank_closed_start = pd.to_datetime('2023-03-10 22:00:00').tz_localize('UTC')
    bank_closed_end = pd.to_datetime('2023-03-13 13:30:00').tz_localize('UTC')
    ax.axvspan(bank_closed_start, bank_closed_end, color='gray', alpha=0.25, label='TradFi Off-Hours (Weekend Basin)')

    depeg_peak = pd.to_datetime('2023-03-11 08:00:00').tz_localize('UTC')
    ax.axvline(depeg_peak, color='red', linestyle='--', linewidth=2, label='USDC Depeg Peak')

    ax.legend(loc='upper left', fontsize=11)

    plt.tight_layout()
    plt.savefig("DeepDive_Intraday_VolumeBars.png", dpi=300)
    print("plot saved to DeepDive_Intraday_VolumeBars.png")
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import yfinance as yf
import os

METRICS_PATH = "./Data/Processed_Metrics_15min"
file_name = 'metrics_agg_15min_std_kraken_btc_usd_march_2023.csv'
file_path = os.path.join(METRICS_PATH, file_name)

df = pd.read_csv(file_path, index_col=0, parse_dates=True)
if df.index.tz is None:
    df.index = df.index.tz_localize('UTC')
else:
    df.index = df.index.tz_convert('UTC')

df_reg = df.loc['2023-03-01':'2023-03-15'].copy()

vix = yf.download('^VIX', start='2023-03-01', end='2023-03-16')
vix_close = vix['Close'].copy()
vix_close.index = vix_close.index.tz_localize('UTC')
df_reg['vix'] = vix_close.reindex(df_reg.index, method='ffill').bfill()

if 'close' in df_reg.columns:
    df_reg['abs_ret'] = np.abs(np.log(df_reg['close'] / df_reg['close'].shift(1)))
else:
    df_reg['abs_ret'] = np.sqrt(df_reg['rv'])

df_reg['cs_lag1'] = df_reg['cs_proxy'].shift(1)
df_reg['log_volume'] = np.log(df_reg['dollar_volume'] + 1)

df_reg = df_reg.replace([np.inf, -np.inf], np.nan)
df_reg = df_reg.dropna(subset=['cs_proxy', 'cs_lag1', 'abs_ret', 'log_volume', 'rv', 'vix'])

Y1 = df_reg['cs_proxy']
X1 = df_reg[['cs_lag1', 'abs_ret', 'rv', 'log_volume', 'vix']]
X1 = sm.add_constant(X1)

model_final = sm.OLS(Y1, X1).fit()
print(model_final.summary())